# Building Agents That Use Code — 데스크탑 Ollama판

코스 섹션 [CodeAgents](https://huggingface.co/learn/agents-course/unit2/smolagents/code_agents)의 예제를 **내 데스크탑 Ollama(`qwen3.5:9b`)**로 한 칸씩 돌려 봐요.
노트(`../notes.md` 3번)에서 읽은 내용을 **눈으로 확인**하는 게 목표예요.

| 단계 | 하는 일 | 노트에서 읽은 것 |
|---|---|---|
| 0~1 | 준비 확인, 모델 연결 | 모델 클래스 (Why Use 섹션) |
| 2 | 예제 ① 플레이리스트 — 기본 도구(웹 검색) | `web_search(...)` 코드가 실행 로그에 보임 |
| 3 | **속 들여다보기** — 기억(memory)과 messages | `SystemPromptStep` · `TaskStep` · `ActionStep`, `write_memory_to_messages()` |
| 4 | 예제 ② 메뉴 — 내 도구(`@tool`) | docstring의 허용 값이 환각을 줄임 |
| 5 | 예제 ③ 준비 시간 — `import` 허락 + `step_callbacks` | 허용 목록, `additional_authorized_imports`, 바퀴마다 부르는 내 함수 |
| 6 | 완성판 Alfred — 도구 만드는 세 가지 길 | 기본 도구 · `@tool` · `Tool` 클래스, `max_steps`, `verbosity_level` |
| 7 | Hub 공유 · Langfuse | **실행 안 함** (이유는 그 칸에) |

**원문과 다른 점**: 원문은 `InferenceClientModel()`(HF 서버, 크레딧 필요) → 여기선 `LiteLLMModel("ollama_chat/qwen3.5:9b")`. **모델 칸만 바꾸고 나머지 코드는 원문 그대로**예요.

위에서부터 `Shift + Enter`로 한 칸씩 실행하세요. 모델이 9B(작은 편)라서 원문 결과와 다르게 움직일 수 있어요 — 그것도 관찰 거리예요.

## 0단계: 준비 확인

Ollama가 켜져 있는지, 이 venv에 필요한 게 다 깔렸는지 봐요.

In [ ]:
import importlib.util
import json
import urllib.request

import smolagents

with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=10) as resp:
    models = [m["name"] for m in json.loads(resp.read())["models"]]

print("smolagents 버전:", smolagents.__version__)
print("qwen3.5:9b 있음?", "qwen3.5:9b" in models)
for pkg in ["litellm", "ddgs", "markdownify", "requests"]:
    print(f"{pkg} 설치됨?", importlib.util.find_spec(pkg) is not None)

✅ **확인할 것**: `smolagents 버전: 1.26.0`, 나머지 전부 `True`
- `ModuleNotFoundError: No module named 'smolagents'` → venv를 켜지 않고 Jupyter를 열었거나 `pip install -r requirements.txt`를 안 함
- `ConnectionRefusedError` → Ollama 앱이 꺼져 있음 (시작 메뉴에서 Ollama 실행)
- `False`가 있으면 → PowerShell에서 `pip install -r requirements.txt` 다시

## 1단계: 모델 연결

Unit 1 `smolagents_first_agent.ipynb`와 **같은 설정**이에요.

| 설정 | 뜻 |
|---|---|
| `model_id="ollama_chat/qwen3.5:9b"` | LiteLLM(통역기)에게 "Ollama에 채팅 방식으로" |
| `num_ctx=8192` | 한 번에 읽는 길이. smolagents system prompt가 길어서 넉넉히 |
| `reasoning_effort="none"` | 속생각(thinking) 끄기 → 빨라짐 |

In [ ]:
from smolagents import LiteLLMModel

model = LiteLLMModel(
    model_id="ollama_chat/qwen3.5:9b",
    api_base="http://localhost:11434",
    num_ctx=8192,
    reasoning_effort="none",
)

reply = model([{"role": "user", "content": [{"type": "text", "text": "Say hello in one short sentence."}]}])
print(reply.content)

✅ **확인할 것**: 짧은 인사말. 첫 호출은 모델을 GPU에 올리느라 수십 초 걸릴 수 있어요.

## 2단계: 예제 ① 파티 플레이리스트 — 기본 도구(웹 검색)

원문 코드 그대로예요. 바뀐 건 `model=InferenceClientModel()` → `model=model`(1단계)뿐.
- `DuckDuckGoSearchTool()` = smolagents가 미리 만들어 둔 웹 검색 도구. 모델에게 보이는 이름은 **`web_search`**
- 원문엔 없지만 `max_steps=6`을 넣었어요: 작은 모델이 헤매도 6바퀴에서 멈추게 (기본값은 20)
- 진짜 인터넷 검색이라 데스크탑 인터넷이 필요해요 (토큰은 불필요)

In [ ]:
from smolagents import CodeAgent, DuckDuckGoSearchTool

agent = CodeAgent(tools=[DuckDuckGoSearchTool()], model=model, max_steps=6)

answer = agent.run("Search for the best music recommendations for a party at the Wayne's mansion.")
print("\n★ 최종 답:", answer)

✅ **확인할 것**
- 원문처럼 `─ Executing parsed code: ─` 상자 안에 `web_search(query="...")` + `print(...)` 코드가 보임 = **모델이 도구를 함수처럼 부르는 코드**를 씀
- 그 아래 `Execution logs:` = `print`로 찍힌 검색 결과 = **관찰(Observation)**. 다음 바퀴에 모델이 이걸 읽음
- 몇 바퀴(`Step 1`, `Step 2` …) 만에 `final_answer(...)`로 끝났는지
- `RatelimitException`·`DDGSException` = 검색 사이트가 잠깐 막은 것 → 1~2분 뒤 이 칸만 다시 실행. 에러가 관찰로 돌아가 모델이 검색어를 바꿔 재시도하는 모습도 볼 수 있어요 (Unit 1 실습 때처럼)

## 3단계: 속 들여다보기 — `CodeAgent.run()` 안에서 일어난 일

원문의 "How Does a Code Agent Work?" 그림을 **방금 돈 에이전트의 기억(memory)**으로 확인해요.

### 3-A. 일지(memory)에 쌓인 단계들

In [ ]:
print("맨 앞:", type(agent.memory.system_prompt).__name__, f"(system prompt {len(agent.memory.system_prompt.system_prompt)}자)")
for step in agent.memory.steps:
    name = type(step).__name__
    if name == "TaskStep":
        print(f"{name}: {step.task[:80]}")
    elif name == "ActionStep":
        sec = step.timing.duration or 0
        print(f"{name} {step.step_number}바퀴: {sec:.1f}초, 최종 답 바퀴? {step.is_final_answer}")
        print("    실행한 코드:", (step.code_action or "").replace("\n", " ⏎ ")[:150])
        if step.error:
            print("    에러:", str(step.error)[:150])
    else:
        print(name)

✅ **확인할 것** — 원문 설명과 1:1로 맞춰 보세요

| 원문 | 위 출력에서 |
|---|---|
| 1. system prompt → `SystemPromptStep` | 맨 앞 줄 |
| 1. 사용자 질문 → `TaskStep` | `TaskStep: Search for the best music ...` |
| 2.5 바퀴마다 결과 → `ActionStep` | `ActionStep 1바퀴`, `2바퀴` … 마지막은 `최종 답 바퀴? True` |

(`PlanningStep`이 보이면 계획 단계예요. 이 예제에선 안 켰으니 보통 안 나와요)

### 3-B. `write_memory_to_messages()` — 일지를 LLM용 messages로 바꾸기

원문 2.1단계. 매 바퀴 모델에게 **실제로 보내는 대화 목록**이 이거예요. Unit 1 Dummy Agent에서 우리가 `messages` 리스트를 손으로 만들어 보낸 것과 같아요.

In [ ]:
msgs = agent.write_memory_to_messages()
print("messages 개수:", len(msgs), "\n")
for i, m in enumerate(msgs):
    role = m.role.value if hasattr(m.role, "value") else m.role
    text = m.content if isinstance(m.content, str) else " ".join(c.get("text", "") for c in m.content if isinstance(c, dict))
    print(f"[{i}] {role:<13} | {text.strip()[:100].replace(chr(10), ' ')} ...")

# 모델에게 보내기 직전, smolagents가 역할 이름을 바꿔 줌
from smolagents.models import tool_role_conversions
print()
for before, after in tool_role_conversions.items():
    print(f"보낼 때 {before.value} → {after.value}")

✅ **확인할 것**
- `[0] system` = system prompt (도구 설명 + "Thought → `<code>` → Observation 하라"는 규칙)
- `[1] user` = `New task: ...` 질문
- 그 뒤로 한 바퀴에 3개씩 **번갈아** 쌓임

| 역할 | 내용 | TAO에서 |
|---|---|---|
| `assistant` | 모델이 쓴 생각 + `<code>` | Thought + Action |
| `tool-call` | "이 코드를 `python_interpreter`로 실행했다"는 기록 | Action (실행 기록) |
| `tool-response` | `Observation: ...` 또는 `Error: ...` 실행 결과 | Observation |

- `tool-call`·`tool-response`는 smolagents 안에서만 쓰는 이름이에요. **모델에게 보내기 직전** 맨 아래 출력처럼 `tool-call → assistant`, `tool-response → user`로 바뀌어요
- 결국 관찰은 **user 역할**로 들어가요 = Unit 1 Dummy 6-B에서 우리가 고친 방향과 같아요
- 바퀴가 늘수록 이 목록이 길어짐 → 매번 전부 다시 보냄 (Unit 1에서 본 Input tokens 누적)

## 4단계: 예제 ② 메뉴 — 내 도구 만들기 (`@tool`)

원문 코드 그대로(모델만 교체). 핵심은 **docstring에 허용 값(Allowed values)을 적어 둔 것**이에요.
모델은 함수 속 `if` 코드를 못 보고, **docstring으로 만든 설명서만** 봐요.

In [ ]:
from smolagents import CodeAgent, tool


# Tool to suggest a menu based on the occasion
@tool
def suggest_menu(occasion: str) -> str:
    """
    Suggests a menu based on the occasion.
    Args:
        occasion (str): The type of occasion for the party. Allowed values are:
                        - "casual": Menu for casual party.
                        - "formal": Menu for formal party.
                        - "superhero": Menu for superhero party.
                        - "custom": Custom menu.
    """
    if occasion == "casual":
        return "Pizza, snacks, and drinks."
    elif occasion == "formal":
        return "3-course dinner with wine and dessert."
    elif occasion == "superhero":
        return "Buffet with high-energy and healthy food."
    else:
        return "Custom menu for the butler."


# 모델이 보게 될 설명서 (system prompt에 이렇게 들어감)
print(suggest_menu.to_code_prompt())

✅ **확인할 것**: `def suggest_menu(occasion: string) -> string:` 아래에 docstring이 그대로 들어감 → **허용 값 4개가 모델에게 보임**. 함수 속 `if` 코드는 안 보임

이제 원문처럼 일을 시켜요.

In [ ]:
# Alfred, the butler, preparing the menu for the party
agent = CodeAgent(tools=[suggest_menu], model=model, max_steps=6)

# Preparing the menu for the party
answer = agent.run("Prepare a formal menu for the party.")
print("\n★ 최종 답:", answer)

✅ **확인할 것**
- 코드에 `suggest_menu(occasion="formal")` — 허용 값 중 하나를 정확히 씀
- 관찰이 `3-course dinner with wine and dessert.`
- 모델이 `"Formal"`(대문자)이나 `"elegant"` 같은 값을 넣으면 `else`로 빠져 `Custom menu ...`가 나와요 → 그게 원문이 말한 **환각(hallucination)**. 실제로 나왔는지 보세요

### (선택) 실험: 허용 값을 지우면?

같은 함수에서 docstring의 허용 값만 지우고, **목록에 없는 말**로 시켜 봐요. "슈퍼히어로"라는 단어 없이 배트맨 파티를 시키면 모델이 `"superhero"`를 고를 수 있을까요?

In [ ]:
@tool
def suggest_menu_vague(occasion: str) -> str:
    """
    Suggests a menu based on the occasion.
    Args:
        occasion: The type of occasion for the party.
    """
    return suggest_menu(occasion)   # 속 동작은 위와 똑같음


for t in [suggest_menu, suggest_menu_vague]:
    agent = CodeAgent(tools=[t], model=model, max_steps=4, verbosity_level=0)   # 0 = 로그 숨김
    answer = agent.run(f"Use {t.name} to get a menu for a Batman-themed party.")
    used = [s.code_action for s in agent.memory.steps if getattr(s, "code_action", None)]
    print(f"== {t.name}")
    print("   모델이 쓴 코드:", " | ".join(c.replace(chr(10), ' ⏎ ')[:120] for c in used))
    print("   최종 답:", answer, "\n")

✅ **확인할 것**: 허용 값이 있는 쪽은 `"superhero"`를 고르기 쉽고, 없는 쪽은 `"Batman"`, `"batman-themed"` 같은 값을 지어내서 `Custom menu ...`가 나오기 쉬워요. 결과는 실행마다 달라질 수 있어요 — 실제로 어땠는지 기록해 두세요

## 5단계: 예제 ③ 준비 시간 — `import` 허락하기 + `step_callbacks`

### 5-A. 에이전트의 코드는 "내 파이썬"이 아니라 smolagents 실행기에서 돈다

모델이 쓴 코드는 smolagents의 **자체 실행기**(`LocalPythonExecutor`)에서 돌아요. 이 실행기는 **허용 목록에 있는 모듈만** `import`하게 해 줘요.

In [ ]:
from smolagents import CodeAgent

agent = CodeAgent(tools=[], model=model)
agent.python_executor.send_tools(agent.tools)   # 원래 agent.run()이 해 주는 준비(print 같은 기본 함수·도구 등록)를 직접
print("기본 허용 import:", sorted(agent.authorized_imports))

# 실행기에 직접 코드를 넣어 보기 (모델 없이)
for snippet in ["import datetime\nprint(datetime.date.today())", "import os\nprint(os.listdir('.'))"]:
    print("\n▶", snippet.replace("\n", " ⏎ "))
    try:
        out = agent.python_executor(snippet)
        print("   성공:", out.logs.strip())
    except Exception as e:
        print("   막힘:", type(e).__name__, "-", str(e)[:160])

✅ **확인할 것**
- 기본 허용 목록에 **`datetime`이 이미 있음** (`collections`, `math`, `random`, `time` … 도)
- `import datetime` → 성공, 오늘 날짜
- `import os` → **막힘**: `Import of os is not allowed` — 모델이 실수나 장난으로 내 파일을 건드리지 못하게
- 참고: 이건 "허용 목록" 수준의 안전장치예요. 공식 문서도 완벽한 격리는 아니라고 해요 → 진짜 격리는 Docker·E2B 같은 원격 sandbox

### 5-B. `additional_authorized_imports` — 목록 밖 모듈 허락하기

`json`은 기본 목록에 없어요(위 출력 확인). 허락 전/후를 비교해요.

In [ ]:
for extra in [None, ["json"]]:
    a = CodeAgent(tools=[], model=model, additional_authorized_imports=extra)
    a.python_executor.send_tools(a.tools)
    try:
        out = a.python_executor('import json\nprint(json.dumps({"party": "ready"}))')
        print(f"additional_authorized_imports={extra} → 성공:", out.logs.strip())
    except Exception as e:
        print(f"additional_authorized_imports={extra} → 막힘:", str(e)[:100])

✅ **확인할 것**: `None` → 막힘, `['json']` → 성공. 원문 예제의 `['datetime']`은 이미 기본 허용이라 **넣어도 안 넣어도 똑같아요**. `pandas`처럼 목록에 없는 걸 쓸 때 진짜 필요해요

### 5-C. 원문 예제 + `step_callbacks` (바퀴마다 부르는 내 함수)

원문 예제를 그대로 돌리면서, 원문 "How Does a Code Agent Work?" 끝에 나온 **`step_callbacks`**도 붙여 봐요.
`report` 함수를 넘기면 smolagents가 **한 바퀴(ActionStep)가 끝날 때마다** 이 함수를 불러 줘요.

In [ ]:
from smolagents import CodeAgent
# import numpy as np   # ← 원문에 있던 줄. numpy는 이 venv에 없고 필요도 없어서 주석 처리 (아래 설명)
import time
import datetime


def report(step, agent=None):
    """한 바퀴가 끝날 때마다 smolagents가 불러 주는 함수"""
    sec = step.timing.duration or 0
    tokens = step.token_usage.input_tokens if step.token_usage else "?"
    print(f"\n📋 [callback] {step.step_number}바퀴 끝 — {sec:.1f}초, 입력 토큰 {tokens}, 에러 {'있음' if step.error else '없음'}\n")


agent = CodeAgent(
    tools=[],
    model=model,
    additional_authorized_imports=["datetime"],
    step_callbacks=[report],
    max_steps=6,
)

answer = agent.run(
    """
    Alfred needs to prepare for the party. Here are the tasks:
    1. Prepare the drinks - 30 minutes
    2. Decorate the mansion - 60 minutes
    3. Set up the menu - 45 minutes
    4. Prepare the music and playlist - 45 minutes

    If we start right now, at what time will the party be ready?
    """
)
print("\n★ 최종 답:", answer)
print("검산: 지금", datetime.datetime.now().strftime("%H:%M"), "+ 180분 =", (datetime.datetime.now() + datetime.timedelta(minutes=180)).strftime("%H:%M"))

✅ **확인할 것**
- 모델 코드에 `import datetime` + `datetime.datetime.now()` + `timedelta(minutes=...)` → **도구 없이 코드로 계산** (원문 장점 "Generality")
- 30+60+45+45 = **180분** → 마지막 줄 검산과 같은 시각이면 정답. (모델이 "순서대로 하나씩"으로 더했는지 보세요. 일을 동시에 한다고 가정하면 답이 달라질 수 있어요)
- 바퀴마다 `📋 [callback] ...` 줄이 찍힘 = `step_callbacks`가 동작. 입력 토큰이 바퀴마다 **늘어나는** 것도 보세요 (3-B의 messages가 길어지니까)
- 위쪽 `import numpy`(주석 처리), `import time`, `import datetime`은 **내 노트북의 import**예요. 에이전트 코드의 허용과는 상관없어요 (그건 5-A의 허용 목록이 정함)

## 6단계: 완성판 Alfred — 도구 만드는 세 가지 길

원문의 큰 코드 그대로예요(모델만 교체). 도구 6개가 세 종류로 나뉘어요.

| 종류 | 도구 | 모양 |
|---|---|---|
| 기본 제공 | `DuckDuckGoSearchTool`, `VisitWebpageTool`(웹페이지 열어 읽기), `FinalAnswerTool` | 가져다 넣기만 |
| `@tool` 함수 | `suggest_menu`, `catering_service_tool` | 함수 + 타입 힌트 + docstring |
| `Tool` 클래스 | `SuperheroPartyThemeTool` | `name` · `description` · `inputs` · `output_type`을 손으로 채우고, 실제 일은 `forward()` |

In [ ]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, FinalAnswerTool, Tool, tool, VisitWebpageTool


@tool
def suggest_menu(occasion: str) -> str:
    """
    Suggests a menu based on the occasion.
    Args:
        occasion: The type of occasion for the party.
    """
    if occasion == "casual":
        return "Pizza, snacks, and drinks."
    elif occasion == "formal":
        return "3-course dinner with wine and dessert."
    elif occasion == "superhero":
        return "Buffet with high-energy and healthy food."
    else:
        return "Custom menu for the butler."


@tool
def catering_service_tool(query: str) -> str:
    """
    This tool returns the highest-rated catering service in Gotham City.

    Args:
        query: A search term for finding catering services.
    """
    # Example list of catering services and their ratings
    services = {
        "Gotham Catering Co.": 4.9,
        "Wayne Manor Catering": 4.8,
        "Gotham City Events": 4.7,
    }

    # Find the highest rated catering service (simulating search query filtering)
    best_service = max(services, key=services.get)

    return best_service


class SuperheroPartyThemeTool(Tool):
    name = "superhero_party_theme_generator"
    description = """
    This tool suggests creative superhero-themed party ideas based on a category.
    It returns a unique party theme idea."""

    inputs = {
        "category": {
            "type": "string",
            "description": "The type of superhero party (e.g., 'classic heroes', 'villain masquerade', 'futuristic Gotham').",
        }
    }

    output_type = "string"

    def forward(self, category: str):
        themes = {
            "classic heroes": "Justice League Gala: Guests come dressed as their favorite DC heroes with themed cocktails like 'The Kryptonite Punch'.",
            "villain masquerade": "Gotham Rogues' Ball: A mysterious masquerade where guests dress as classic Batman villains.",
            "futuristic gotham": "Neo-Gotham Night: A cyberpunk-style party inspired by Batman Beyond, with neon decorations and futuristic gadgets.",
        }

        return themes.get(category.lower(), "Themed party idea not found. Try 'classic heroes', 'villain masquerade', or 'futuristic Gotham'.")


# Alfred, the butler, preparing the menu for the party
agent = CodeAgent(
    tools=[
        DuckDuckGoSearchTool(),
        VisitWebpageTool(),
        suggest_menu,
        catering_service_tool,
        SuperheroPartyThemeTool(),
        FinalAnswerTool(),
    ],
    model=model,
    max_steps=10,
    verbosity_level=2,
)

print("도구 목록:", list(agent.tools))
print()
# @tool이 자동으로 만든 설명서 vs Tool 클래스에 손으로 쓴 설명서 — 모델에게는 똑같은 모양으로 보임
print(catering_service_tool.to_code_prompt())
print(agent.tools["superhero_party_theme_generator"].to_code_prompt())

✅ **확인할 것**
- 도구 목록에 6개 (`web_search`, `visit_webpage`, `suggest_menu`, `catering_service_tool`, `superhero_party_theme_generator`, `final_answer`)
- `@tool` 함수와 `Tool` 클래스의 설명서가 **같은 꼴**(`def 이름(인자: string) -> string:` + 설명)로 나옴 → 만드는 방법만 다르고 모델에겐 똑같이 보여요
- `FinalAnswerTool()`은 원문에 있어서 넣었지만, CodeAgent는 `final_answer`를 **원래 자동으로** 넣어요 (빼도 목록이 같음)

### 6-A. 원문 질문 그대로

In [ ]:
answer = agent.run("Give me the best playlist for a party at the Wayne's mansion. The party idea is a 'villain masquerade' theme")
print("\n★ 최종 답:", answer)

✅ **확인할 것**
- `verbosity_level=2` → 2단계보다 로그가 훨씬 자세함 (모델에게 보낸 내용까지 보일 수 있음)
- 도구 6개 중 **어떤 걸 골랐나?** 플레이리스트 질문이라 `web_search` 위주일 거예요. `superhero_party_theme_generator("villain masquerade")`도 불렀는지 보세요
- `visit_webpage`로 검색 결과 링크를 열어 읽었는지 (바퀴가 길어질 수 있음, 최대 10바퀴)

### 6-B. 직접 만든 도구들을 쓰게 하는 질문

6-A는 웹 검색만으로 끝나기 쉬워요. 직접 만든 세 도구가 다 쓰이도록 질문을 바꿔 봐요 (원문에 없는 추가 실험).

In [ ]:
answer = agent.run(
    "Plan a 'villain masquerade' party: get a theme idea, a formal menu, and the best catering service in Gotham. "
    "Do not search the web. Summarize the plan in 3 bullet points."
)
print("\n★ 최종 답:", answer)

✅ **확인할 것**
- 한 바퀴 코드 안에서 세 도구를 **한꺼번에** 불렀는지 (`theme = ...`, `menu = ...`, `catering = ...` → 원문 장점 "Composability")
- `catering_service_tool`은 `query`를 뭘 넣든 `Gotham Catering Co.`를 돌려줘요 — 연습용 가짜 도구라서
- 6단계의 `suggest_menu`는 4단계와 달리 **허용 값이 docstring에 없어요**. 모델이 `"formal"`을 정확히 넣었나요?

## 7단계: Hub 공유 · Langfuse 추적 — 이번엔 실행 안 함

| 원문 기능 | 안 하는 이유 | 나중에 하려면 |
|---|---|---|
| `agent.push_to_hub('아이디/AlfredAgent')` | HF 계정에 **쓰기**(공개 저장소가 생김) + 쓰기 권한 토큰 필요 | AI에게 요청 → 승인 표 거쳐서 |
| `agent.from_hub(..., trust_remote_code=True)` | **남이 올린 코드를 내 PC에서 실행**하는 것 | 믿을 수 있는 저장소인지 확인 후, 승인 거쳐서 |
| Langfuse + OpenTelemetry | Langfuse 가입 + 비밀 키 필요. 원문처럼 키를 코드에 쓰면 안 됨 → `.env`에만 | Bonus Unit 2에서. 내 PC에서만 보는 방법도 있음 ([inspect runs](https://huggingface.co/docs/smolagents/tutorials/inspect_runs)) |

대신 **3단계의 `agent.memory`와 5-C의 `step_callbacks`**가 추적의 가장 단순한 형태예요. Langfuse는 이 기록을 바깥 서비스로 보내서 예쁜 화면으로 보여 주는 것이라고 생각하면 돼요.

## 다 했으면

각 단계에서 이런 걸 복사해서 AI에게 붙여 주세요 → 노트 3번 "실습 결과"에 기록해요.
- 2단계: 몇 바퀴, 모델이 쓴 검색 코드, 최종 답 (에러가 있었다면 그 내용)
- 3-B: messages 개수와 역할 순서 (`assistant` → `tool-call` → `tool-response` 반복인지)
- 4단계 (선택 실험 포함): 모델이 넣은 `occasion` 값
- 5-C: 최종 답 시각 vs 검산 시각, callback 줄의 입력 토큰 변화
- 6-A · 6-B: 어떤 도구를 골랐나, 한 바퀴에 몇 개 불렀나

Jupyter 끄기: 브라우저 탭을 닫고 → PowerShell 창에서 `Ctrl + C` 두 번 (안 되면 PowerShell 창을 X로 닫아도 돼요).